# <center>Pre processing</center>

*Aplica as etapas de pré-processamento*

---

In [1]:
!pip install nltk spacy unidecode
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 44.4 MB/s  0:00:00m0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import pandas as pd
from nltk.corpus import stopwords
import nltk
import numpy as np
import os
import sys
from nltk.corpus import stopwords
nltk.download('stopwords')
import spacy as sp
from PreProcessing.pre_processing import PreProcessing
from tqdm import tqdm


project_root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root_path not in sys.path:
    sys.path.append(project_root_path)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/students/moliveira/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
def remove_repetion_caracteres(string, max_repetition=2):
    if not string:
        return string
    
    result = string[0]
    count = 1
    
    for i in range(1, len(string)):
        if string[i] == string[i-1]:
            count += 1
            if count <= max_repetition:
                result += string[i]
        else:
            count = 1
            result += string[i]
    
    return result

def preprocess_text_pipeline(#input_csv_path='./data/dataFrame.csv', 
                              #output_csv_path='./data/dataFrame.csv',
                              df,
                              stopwords_file='stopwords.txt',
                              text_column="comments"):
   
    stem = sp.load("en_core_web_sm")
    pp = PreProcessing(language="en")
    
    custom_stopwords = [line.strip() for line in open(stopwords_file, 'r', encoding='utf-8')]
    english_stopwords = set(stopwords.words('english'))
    
    # Adiciona stopwords à lista da classe PreProcessing
    pp.append_stopwords_list(list(english_stopwords - set(pp.stopwords)) + custom_stopwords)

    def preprocessing(text):
        if pd.isna(text):
            return np.nan

        tokens = stem(text.lower()) # Processo de lematização da biblioteca spaCy - retorna a lista dos tokens do texto
        text = ' '.join([text for token in tokens for text in token.lemma_.strip().split()]) # Junta estes tokens na ordem do texto bruto
        text = pp.remove_stopwords(text) # Remove stopwords presentes
        text = pp.lowercase_unidecode(text) # Coloca tudo em lowercase e remove acento
        text = pp.remove_stopwords(text) # Remove stopwords presentes 
        text = pp.remove_tweet_marking(text) # Remove @ ou # seguido de 1 ou mais carcteres e n ' ' seguidos
        text = remove_repetion_caracteres(text) # Remove a repetição de caracteres ex: gooool -> gool
        text = pp.remove_urls(text) # Remove http\S+ *, ou seja, qualquer http seguido de 1 ou mais caracteres e os espaços no final
        text = pp.remove_punctuation(text) # Remove os sinais de pontuação e reorganiza os espaços
        text = pp.remove_numbers(text) # Remove os números
        text = pp.remove_n(text, n=3) # Remove palavras de tamanho <= n(n=3)
        
        return text
    
    tqdm.pandas()
    df['clean_text'] = df[text_column].progress_apply(preprocessing)

    return df

In [4]:
df = pd.read_csv("data/english_videos_infos.csv")
display(df)

,video_id,title,description,title_description,lang
0,jEKzQV5oajY,Free bus travel for migrants scrapped. For 5 m...,NaN,Free bus travel for migrants scrapped. For 5 m...,en
1,xrGGce8cmx8,What is Spiritual Warfare?,"This charge I commit unto thee, son Timothy, a...",What is Spiritual Warfare? This charge I commi...,en
2,uaozGpSc4nc,80 Putins Have Layers,"In February 2024, Tucker Carlson stood in fron...","80 Putins Have Layers In February 2024, Tucker...",en
3,A59ftbhsQUE,𝐆𝐀𝐋𝐀𝐂𝐓𝐈𝐂 𝐀𝐋𝐋𝐈𝐀𝐍𝐂𝐄 𝐌𝐄𝐒𝐒𝐀𝐆𝐄|𝐁𝐈𝐆 𝐍𝐄𝐖𝐒!~𝐑𝐄𝐌𝐀𝐈𝐍 𝐀𝐋𝐄...,#pleiadians #5dimension #ashtar\n#pleiadians #...,𝐆𝐀𝐋𝐀𝐂𝐓𝐈𝐂 𝐀𝐋𝐋𝐈𝐀𝐍𝐂𝐄 𝐌𝐄𝐒𝐒𝐀𝐆𝐄|𝐁𝐈𝐆 𝐍𝐄𝐖𝐒!~𝐑𝐄𝐌𝐀𝐈𝐍 𝐀𝐋𝐄...,en
4,AKU0RokegSo,Los Angeles rain: Studio City homes evacuated ...,An atmospheric river storm has caused mudslide...,Los Angeles rain: Studio City homes evacuated ...,en
...,...,...,...,...,...
686621,xV2G3GFf2ug,Did you know this ? #gaza #god #israel #jesus ...,Have you seen this ? \n\nFULL VIDEO: Aries int...,Did you know this ? #gaza #god #israel #jesus ...,en
686622,GEhEUy85PsY,Meet the balkans,This is a video,Meet the balkans This is a video,en
686623,fNTfEPLoAik,Former Clark County sheriff arrested on 15 fel...,Former Clark County sheriff arrested on 15 fel...,Former Clark County sheriff arrested on 15 fel...,en
686624,8pEHwRiJVJM,What’s your thougts? #Bible #god #Christianscr...,What’s your thoughts?\n\nFULL VIDEO: 👇🏽\nhttps...,What’s your thougts? #Bible #god #Christianscr...,en


In [5]:
df_clean = preprocess_text_pipeline(df=df, text_column='title_description')
display(df_clean)
df_clean.to_csv("data/preprocessed_english_titles")

100%|██████████| 686626/686626 [15:14:09<00:00, 12.52it/s]   


,video_id,title,description,title_description,lang,clean_text
0,jEKzQV5oajY,Free bus travel for migrants scrapped. For 5 m...,NaN,Free bus travel for migrants scrapped. For 5 m...,en,free travel migrant scrap minute
1,xrGGce8cmx8,What is Spiritual Warfare?,"This charge I commit unto thee, son Timothy, a...",What is Spiritual Warfare? This charge I commi...,en,spiritual warfare charge commit unto thee timo...
2,uaozGpSc4nc,80 Putins Have Layers,"In February 2024, Tucker Carlson stood in fron...","80 Putins Have Layers In February 2024, Tucker...",en,putin layer february tucker carlson stand onio...
3,A59ftbhsQUE,𝐆𝐀𝐋𝐀𝐂𝐓𝐈𝐂 𝐀𝐋𝐋𝐈𝐀𝐍𝐂𝐄 𝐌𝐄𝐒𝐒𝐀𝐆𝐄|𝐁𝐈𝐆 𝐍𝐄𝐖𝐒!~𝐑𝐄𝐌𝐀𝐈𝐍 𝐀𝐋𝐄...,#pleiadians #5dimension #ashtar\n#pleiadians #...,𝐆𝐀𝐋𝐀𝐂𝐓𝐈𝐂 𝐀𝐋𝐋𝐈𝐀𝐍𝐂𝐄 𝐌𝐄𝐒𝐒𝐀𝐆𝐄|𝐁𝐈𝐆 𝐍𝐄𝐖𝐒!~𝐑𝐄𝐌𝐀𝐈𝐍 𝐀𝐋𝐄...,en,GALACTIC ALLIANCE MESSAGE NEWS REMAIN ALERT PR...
4,AKU0RokegSo,Los Angeles rain: Studio City homes evacuated ...,An atmospheric river storm has caused mudslide...,Los Angeles rain: Studio City homes evacuated ...,en,angeles rain studio city home evacuate mudslid...
...,...,...,...,...,...,...
686621,xV2G3GFf2ug,Did you know this ? #gaza #god #israel #jesus ...,Have you seen this ? \n\nFULL VIDEO: Aries int...,Did you know this ? #gaza #god #israel #jesus ...,en,know gaza israel jesus bible palestine christ ...
686622,GEhEUy85PsY,Meet the balkans,This is a video,Meet the balkans This is a video,en,meet balkan video
686623,fNTfEPLoAik,Former Clark County sheriff arrested on 15 fel...,Former Clark County sheriff arrested on 15 fel...,Former Clark County sheriff arrested on 15 fel...,en,clark county sheriff arrest felony count clark...
686624,8pEHwRiJVJM,What’s your thougts? #Bible #god #Christianscr...,What’s your thoughts?\n\nFULL VIDEO: 👇🏽\nhttps...,What’s your thougts? #Bible #god #Christianscr...,en,thougts bible christianscripture jesus biblica...
